In [ ]:
# !aws s3 sync s3://translinkdata ../data/lambda_sync

In [2]:
import pandas as pd
import numpy as np
import glob
import polars as pl
import os
import hashlib
from datetime import datetime
import altair as alt

# alt.data_transformers.enable("vegafusion")

### Read Data

In [3]:
def read_translink_s3_data_pandas(data_path, pattern):
    """Read local parquet files based on path and pattern.

    Args:
        data_path (str): Local path to the directory containing parquet files
        pattern (str): Either 'position' or 'realtime'

    Returns:
        pd.DataFrame: Combined data from all matching parquet files
    """
    # Create full pattern and print it for debugging
    glob_pattern = os.path.join(data_path, f"*_{pattern}.parquet")
    print(f"Searching with pattern: {glob_pattern}")

    # List all files in directory for debugging
    print(f"Files in directory: {os.listdir(data_path)}")

    # Find all parquet files matching the pattern
    matching_files = glob.glob(glob_pattern)
    print(f"Found files: {matching_files}")

    if not matching_files:
        raise ValueError(f"No parquet files found matching pattern '{pattern}'")

    # Read and combine all matching parquet files
    return pd.concat([pd.read_parquet(f) for f in matching_files], ignore_index=True)

In [4]:
# Using '../' to go up one directory level from notebooks/
# realtime_df = read_translink_s3_data('../data/lambda_sync/raw_data','realtime')

In [5]:
stops = pl.scan_csv("../data/gtfs_static/stops.txt")
trips = pl.scan_csv(
    "../data/gtfs_static/trips.txt",
    infer_schema=True,
    schema_overrides={"route_id": pl.Utf8},
)
calendar = pl.scan_csv(
    "../data/gtfs_static/calendar.txt",
)
stop_times = pl.scan_csv(
    "../data/gtfs_static/stop_times.txt",
).rename({'arrival_time':'scheduled_arrival_time','departure_time':'scheduled_departure_time'})
routes = pl.scan_csv(
    "../data/gtfs_static/routes.txt", schema_overrides={"route_id": pl.Utf8}
)

In [6]:
# stop_times.collect()

In [7]:
trips_enriched = trips.join(calendar, on="service_id", how="left", coalesce=True).join(
    routes, on="route_id", how="left", coalesce=True
)
stop_times_enriched = stop_times.join(
    stops, on="stop_id", how="left", coalesce=True
).join(trips_enriched, on="trip_id", how="left", coalesce=True)


stop_times_enriched.head().collect()

trip_id,scheduled_arrival_time,scheduled_departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_lat,wheelchair_boarding,stop_code,stop_lon,stop_url,parent_station,stop_desc,stop_name,location_type,zone_id,block_id,bikes_allowed,route_id,wheelchair_accessible,direction_id,trip_headsign,shape_id,service_id,trip_short_name,start_date,end_date,monday,tuesday,wednesday,thursday,friday,saturday,sunday,route_long_name,route_type,route_text_color,route_color,agency_id,route_url,route_desc,route_short_name
i64,str,str,i64,i64,str,i64,str,f64,i64,f64,i64,i64,f64,str,i64,str,str,i64,str,str,i64,str,i64,i64,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,str,str
14305517,"""20:02:00""","""20:02:00""",12238,1,null,null,null,null,1,49.285743,1,60095,-122.791623,null,99932,null,"""Lafarge Lake-Douglas Station @…",0,"""ZN 3""","""b_2120930""",1,"""30052""",0,1,"""Millennium Line To VCC-Clark""",298080,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Millennium Line""",1,"""333333""","""ffcd00""","""TL""",null,null,null
14305517,"""20:03:00""","""20:03:00""",12236,2,null,null,null,0.6214,1,49.280417,1,60093,-122.794097,null,99931,null,"""Lincoln Station @ Platform 1""",0,"""ZN 3""","""b_2120930""",1,"""30052""",0,1,"""Millennium Line To VCC-Clark""",298080,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Millennium Line""",1,"""333333""","""ffcd00""","""TL""",null,null,null
14305517,"""20:05:00""","""20:05:00""",12234,3,null,null,null,1.4993,1,49.274919,1,60091,-122.800551,null,99930,null,"""Coquitlam Central Station @ Pl…",0,"""ZN 3""","""b_2120930""",1,"""30052""",0,1,"""Millennium Line To VCC-Clark""",298080,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Millennium Line""",1,"""333333""","""ffcd00""","""TL""",null,null,null
14305517,"""20:08:00""","""20:08:00""",12232,4,null,null,null,3.5732,1,49.277294,1,60089,-122.828164,null,99929,null,"""Inlet Centre Station @ Platfor…",0,"""ZN 3""","""b_2120930""",1,"""30052""",0,1,"""Millennium Line To VCC-Clark""",298080,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Millennium Line""",1,"""333333""","""ffcd00""","""TL""",null,null,null
14305517,"""20:10:00""","""20:10:00""",12230,5,null,null,null,4.867,1,49.27798,1,60087,-122.845609,null,99928,null,"""Moody Centre Station @ Platfor…",0,"""ZN 3""","""b_2120930""",1,"""30052""",0,1,"""Millennium Line To VCC-Clark""",298080,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Millennium Line""",1,"""333333""","""ffcd00""","""TL""",null,null,null


In [8]:
def generate_file_id(filename):
    """Generate a unique ID for a file using SHA-256 hash."""
    return hashlib.sha256(os.path.basename(filename).encode()).hexdigest()[:16]


def read_translink_realtime_polars():
    """Read local parquet files based on path and pattern.

    Args:
        data_path (str): Local path to the directory containing parquet files
        pattern (str): Either 'position' or 'realtime'

    Returns:
        pd.DataFrame: Combined data from all matching parquet files
    """
    data_path = "../data/lambda_sync/raw_data"
    pattern = "realtime"
    # Create full pattern and print it for debugging
    glob_pattern = os.path.join(data_path, f"*_{pattern}.parquet")
    print(f"Searching with pattern: {glob_pattern}")

    # List all files in directory for debugging
    print(f"Files in directory: {os.listdir(data_path)}")

    # Find all parquet files matching the pattern
    matching_files = glob.glob(glob_pattern)
    print(f"Found files: {matching_files}")

    if not matching_files:
        raise ValueError(f"No parquet files found matching pattern '{pattern}'")

    # Read and combine all matching parquet files

    schema = {
        "id": pl.Utf8,
        "is_deleted": pl.Boolean,
        "trip_id": pl.Utf8,
        "start_date": pl.Utf8,
        "schedule_relationship": pl.Int64,
        "route_id": pl.Utf8,
        "direction_id": pl.Int64,
        "vehicle_id": pl.Utf8,
        "vehicle_label": pl.Utf8,
        "current_datetime": pl.Datetime("ns"),
        "stop_sequence": pl.Int64,
        "stop_id": pl.Utf8,
        "arrival_delay": pl.Int64,
        "arrival_time": pl.Float64,
        "departure_delay": pl.Float64,
        "departure_time": pl.Float64,
        "stop_schedule_relationship": pl.Int64,
    }
    collected_df = pl.concat(
        [
            pl.scan_parquet(f)
            .with_columns([pl.col(col).cast(dtype) for col, dtype in schema.items()])
            .with_columns(scrape_id=pl.lit(generate_file_id(f)))
            for f in matching_files
        ],
    )

    # collected_df = pl.scan_parquet(matching_files, schema=schema)

    formatted_df = (
        collected_df.with_columns(
            pl.from_epoch(pl.col("arrival_time"), time_unit="s")
            .dt.convert_time_zone("America/Vancouver")
            .alias("arrival_time"),
        )
        .with_columns(
            pl.from_epoch(pl.col("departure_time"), time_unit="s")
            .dt.convert_time_zone("America/Vancouver")
            .alias("departure_time")
        )
        .with_columns(
            pl.col("current_datetime")
            .dt.convert_time_zone("America/Vancouver")
            .alias("current_datetime")
        )
        .with_columns(pl.col("current_datetime").dt.date().alias("current_date"))
    ).join(
        stop_times_enriched.with_columns(
            pl.col("trip_id").cast(pl.String), pl.col("stop_id").cast(pl.String)
        ),
        on=["trip_id", "stop_id", "stop_sequence", "direction_id"],
        how="left",
        coalesce=True,
    )
    return formatted_df


read_translink_realtime_polars().head().collect()

Searching with pattern: ../data/lambda_sync/raw_data/*_realtime.parquet
Files in directory: ['translink_20250110_033449_realtime.parquet', 'translink_20250104_163448_position.parquet', 'translink_20250104_225449_realtime.parquet', 'translink_20250108_035949_realtime.parquet', 'translink_20250107_230949_realtime.parquet', 'translink_20250102_114450_realtime.parquet', 'translink_20241229_070449_position.parquet', 'translink_20250103_130449_position.parquet', 'translink_20250101_114448_position.parquet', 'translink_20250109_080448_position.parquet', 'translink_20250112_200449_realtime.parquet', 'translink_20250111_215949_realtime.parquet', 'translink_20250108_164949_position.parquet', 'translink_20241228_110948_realtime.parquet', 'translink_20250111_153948_position.parquet', 'translink_20250109_213449_realtime.parquet', 'translink_20241227_053949_position.parquet', 'translink_20250110_162449_position.parquet', 'translink_20250112_093448_position.parquet', 'translink_20250105_003949_realti

id,is_deleted,trip_id,start_date,schedule_relationship,route_id,direction_id,vehicle_id,vehicle_label,current_datetime,stop_sequence,stop_id,arrival_delay,arrival_time,departure_delay,departure_time,stop_schedule_relationship,scrape_id,current_date,scheduled_arrival_time,scheduled_departure_time,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_lat,wheelchair_boarding,stop_code,stop_lon,stop_url,parent_station,stop_desc,stop_name,location_type,zone_id,block_id,bikes_allowed,route_id_right,wheelchair_accessible,trip_headsign,shape_id,service_id,trip_short_name,start_date_right,end_date,monday,tuesday,wednesday,thursday,friday,saturday,sunday,route_long_name,route_type,route_text_color,route_color,agency_id,route_url,route_desc,route_short_name
str,bool,str,str,i64,str,i64,str,str,"datetime[ns, America/Vancouver]",i64,str,i64,"datetime[μs, America/Vancouver]",f64,"datetime[μs, America/Vancouver]",i64,str,date,str,str,str,i64,str,f64,i64,f64,i64,i64,f64,str,i64,str,str,i64,str,str,i64,str,i64,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,str,str
"""14252592""",false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,1,"""11251""",0,2025-01-09 19:52:00 PST,0.0,2025-01-09 19:52:00 PST,0,"""9ef7096c1db33bd1""",2025-01-09,"""19:52:00""","""19:52:00""",null,null,null,null,1,49.209087,1,61337,-123.116823,null,null,null,"""Marine Drive Station @ Bay 1""",0,"""BUS ZN""","""b_2117689""",1,"""6613""",0,"""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,null,null,"""TL""",null,null,"""003"""
"""14252592""",false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,2,"""2151""",-15,2025-01-09 19:54:50 PST,-15.0,2025-01-09 19:54:50 PST,0,"""9ef7096c1db33bd1""",2025-01-09,"""19:55:05""","""19:55:05""",null,null,null,0.7837,0,49.212198,1,52131,-123.109277,null,null,null,"""Eastbound SW Marine Dr @ Manit…",0,"""BUS ZN""","""b_2117689""",1,"""6613""",0,"""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,null,null,"""TL""",null,null,"""003"""
"""14252592""",false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,3,"""2152""",-34,2025-01-09 19:55:29 PST,-34.0,2025-01-09 19:55:29 PST,0,"""9ef7096c1db33bd1""",2025-01-09,"""19:56:03""","""19:56:03""",null,null,null,1.0315,0,49.21214,1,52132,-123.106006,null,null,null,"""Eastbound SE Marine Dr @ Ontar…",0,"""BUS ZN""","""b_2117689""",1,"""6613""",0,"""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,null,null,"""TL""",null,null,"""003"""
"""14252592""",false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,4,"""142""",-10,2025-01-09 19:57:18 PST,-10.0,2025-01-09 19:57:18 PST,0,"""9ef7096c1db33bd1""",2025-01-09,"""19:57:28""","""19:57:28""",null,null,null,1.3937,0,49.211565,1,50142,-123.102035,null,null,null,"""Northbound Main St @ SE Marine…",0,"""BUS ZN""","""b_2117689""",1,"""6613""",0,"""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,null,null,"""TL""",null,null,"""003"""
"""14252592""",false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,5,"""144""",-28,2025-01-09 19:58:19 PST,-28.0,2025-01-09 19:58:19 PST,0,"""9ef7096c1db33bd1""",2025-01-09,"""19:58:47""","""19:58:47""",null,null,null,1.729,0,49.214623,1,59844,-123.101909,null,null,null,"""Northbound Main St @ E 62 Ave""",0,"""BUS ZN""","""b_2117689""",1,"""6613""",0,"""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250106,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,null,null,"""TL""",null,null,"

In [9]:
def read_weather_data_polars():
    weather_files = glob.glob("../data/lambda_sync/weather_data/*.parquet")
    print(weather_files)

    weather_df = pl.scan_parquet(
        weather_files,
    )
    weather_df = (
        (
            weather_df.rename(lambda x: x.replace(".", ""))
            .with_columns(
                pl.col("nowobsTime")
                .cast(pl.Datetime("ns"))
                .dt.convert_time_zone("America/Vancouver")
            )
            .with_columns(
                pl.col("updateTime")
                .cast(pl.Datetime("ns"))
                .dt.convert_time_zone("America/Vancouver")
            )
            .with_columns(pl.col("nowtemp").cast(pl.Int64))
            .with_columns(pl.col("nowprecip").cast(pl.Float64))
        )
        .with_columns(
            nowobsTime_three_hour_bucket=pl.col("nowobsTime").dt.truncate("3h")
        )
        .select(
            [
                # "updateTime",
                # "nowobsTime",
                "nowobsTime_three_hour_bucket",
                "nowtemp",
                "nowtext",
                "nowprecip",
            ]
        )
    )

    return weather_df


read_weather_data_polars().head().collect()

['../data/lambda_sync/weather_data/weather_20250106_085626.parquet', '../data/lambda_sync/weather_data/weather_20241230_055626.parquet', '../data/lambda_sync/weather_data/weather_20250112_115626.parquet', '../data/lambda_sync/weather_data/weather_20250107_115626.parquet', '../data/lambda_sync/weather_data/weather_20250105_205626.parquet', '../data/lambda_sync/weather_data/weather_20241228_053357.parquet', '../data/lambda_sync/weather_data/weather_20250104_175626.parquet', '../data/lambda_sync/weather_data/weather_20250111_175626.parquet', '../data/lambda_sync/weather_data/weather_20250107_235626.parquet', '../data/lambda_sync/weather_data/weather_20250112_235626.parquet', '../data/lambda_sync/weather_data/weather_20250103_025626.parquet', '../data/lambda_sync/weather_data/weather_20250106_145626.parquet', '../data/lambda_sync/weather_data/weather_20250102_205626.parquet', '../data/lambda_sync/weather_data/weather_20250109_145626.parquet', '../data/lambda_sync/weather_data/weather_20250

nowobsTime_three_hour_bucket,nowtemp,nowtext,nowprecip
"datetime[ns, America/Vancouver]",i64,str,f64
2025-01-05 15:00:00 PST,3,"""Clear""",0.0
2024-12-29 12:00:00 PST,5,"""Partly Cloudy""",2.0
2025-01-11 18:00:00 PST,4,"""Cloudy""",0.1
2025-01-06 18:00:00 PST,1,"""Fog""",0.0
2025-01-05 03:00:00 PST,9,"""Cloudy""",0.0


In [10]:
def read_translink_position_polars():
    """Read local parquet files based on path and pattern.

    Args:
        data_path (str): Local path to the directory containing parquet files
        pattern (str): Either 'position' or 'realtime'

    Returns:
        pd.DataFrame: Combined data from all matching parquet files
    """
    data_path = "../data/lambda_sync/raw_data"
    pattern = "position"
    # Create full pattern and print it for debugging
    glob_pattern = os.path.join(data_path, f"*_{pattern}.parquet")
    print(f"Searching with pattern: {glob_pattern}")

    # List all files in directory for debugging
    print(f"Files in directory: {os.listdir(data_path)}")

    # Find all parquet files matching the pattern
    matching_files = glob.glob(glob_pattern)
    print(f"Found files: {matching_files}")

    if not matching_files:
        raise ValueError(f"No parquet files found matching pattern '{pattern}'")

    schema = {
        "id": pl.Utf8,
        "trip_id": pl.Utf8,
        "start_date": pl.Utf8,
        "schedule_relationship": pl.Int64,
        "route_id": pl.Utf8,
        "direction_id": pl.Int64,
        "vehicle_id": pl.Utf8,
        "vehicle_label": pl.Utf8,
        "latitude": pl.Float64,
        "longitude": pl.Float64,
        "current_stop_sequence": pl.Int64,
        "current_status": pl.Int64,
        "timestamp": pl.Int64,
        "stop_id": pl.Utf8,
        "current_datetime": pl.Datetime("ns"),
    }

    df = pl.scan_parquet(matching_files, schema=schema)

    df = (
        df.with_columns(
            pl.col("current_datetime")
            .dt.convert_time_zone("America/Vancouver")
            .alias("current_datetime")
        )
        .with_columns(current_date=pl.col("current_datetime").dt.date())
        .select(
            [
                "trip_id",
                "current_stop_sequence",
                "latitude",
                "longitude",
                "current_datetime",
                # "current_date",
            ]
        )
        .rename({"latitude": "vehicle_latitude", "longitude": "vehicle_longitude"})
    )

    return df


read_translink_position_polars().head().collect()

Searching with pattern: ../data/lambda_sync/raw_data/*_position.parquet
Files in directory: ['translink_20250110_033449_realtime.parquet', 'translink_20250104_163448_position.parquet', 'translink_20250104_225449_realtime.parquet', 'translink_20250108_035949_realtime.parquet', 'translink_20250107_230949_realtime.parquet', 'translink_20250102_114450_realtime.parquet', 'translink_20241229_070449_position.parquet', 'translink_20250103_130449_position.parquet', 'translink_20250101_114448_position.parquet', 'translink_20250109_080448_position.parquet', 'translink_20250112_200449_realtime.parquet', 'translink_20250111_215949_realtime.parquet', 'translink_20250108_164949_position.parquet', 'translink_20241228_110948_realtime.parquet', 'translink_20250111_153948_position.parquet', 'translink_20250109_213449_realtime.parquet', 'translink_20241227_053949_position.parquet', 'translink_20250110_162449_position.parquet', 'translink_20250112_093448_position.parquet', 'translink_20250105_003949_realti

trip_id,current_stop_sequence,vehicle_latitude,vehicle_longitude,current_datetime
str,i64,f64,f64,"datetime[ns, America/Vancouver]"
"""14121792""",1,49.286499,-123.140701,2025-01-04 08:34:48.859985 PST
"""14010917""",2,49.281483,-123.102264,2025-01-04 08:34:48.859998 PST
"""13998796""",27,49.283852,-123.109154,2025-01-04 08:34:48.860005 PST
"""14000152""",13,49.284184,-123.13707,2025-01-04 08:34:48.860012 PST
"""13997768""",23,49.272484,-123.14505,2025-01-04 08:34:48.860019 PST


In [11]:
realtime_df = read_translink_realtime_polars()
weather_df = read_weather_data_polars()
position_df = read_translink_position_polars()

Searching with pattern: ../data/lambda_sync/raw_data/*_realtime.parquet
Files in directory: ['translink_20250110_033449_realtime.parquet', 'translink_20250104_163448_position.parquet', 'translink_20250104_225449_realtime.parquet', 'translink_20250108_035949_realtime.parquet', 'translink_20250107_230949_realtime.parquet', 'translink_20250102_114450_realtime.parquet', 'translink_20241229_070449_position.parquet', 'translink_20250103_130449_position.parquet', 'translink_20250101_114448_position.parquet', 'translink_20250109_080448_position.parquet', 'translink_20250112_200449_realtime.parquet', 'translink_20250111_215949_realtime.parquet', 'translink_20250108_164949_position.parquet', 'translink_20241228_110948_realtime.parquet', 'translink_20250111_153948_position.parquet', 'translink_20250109_213449_realtime.parquet', 'translink_20241227_053949_position.parquet', 'translink_20250110_162449_position.parquet', 'translink_20250112_093448_position.parquet', 'translink_20250105_003949_realti

### Generate Features

In [12]:
def generate_realtime_features(df: pl.DataFrame):
    """Generate features from realtime data."""
    df = (
        df.with_columns(
            last_arrival_delay=pl.col("arrival_delay")
            .last()
            .over(
                partition_by=["trip_id", "stop_sequence", "current_date"],
                order_by="current_datetime",
            )
        )
        .with_columns(
            last_arrival_delay_time=pl.col("current_datetime")
            .last()
            .over(
                partition_by=["trip_id", "stop_sequence", "current_date"],
                order_by="current_datetime",
            )
        )
        .with_columns(
            last_record_rank=pl.col("current_datetime")
            .rank(descending=True)
            .over(
                partition_by=["trip_id", "stop_sequence", "current_date"],
                order_by="current_datetime",
            )
            .cast(pl.Int64)
        )
        .with_columns(
            hour=pl.col("current_datetime").dt.hour().alias("hour"),
            minute=pl.col("current_datetime").dt.minute().alias("minute"),
        )
    )

    return df

In [13]:
realtime_df_enriched = generate_realtime_features(realtime_df)

### ETL Pipeline

In [14]:
# realtime_df_enriched_sample = realtime_df_enriched.head().collect()

In [15]:
# realtime_df_enriched_sample

In [16]:
def merge_dataframes_etl():
    realtime_selected_columns = [
        "is_deleted",
        "trip_id",
        "start_date",
        "schedule_relationship",
        "route_id",
        "direction_id",
        "vehicle_id",
        "vehicle_label",
        "current_datetime",
        "hour",
        "minute",
        "stop_sequence",
        "stop_id",
        "arrival_delay",
        "last_arrival_delay",
        "last_arrival_delay_time",
        "last_record_rank",
        "arrival_time",
        'scheduled_arrival_time',
        "departure_delay",
        "departure_time",
        'scheduled_departure_time',
        "stop_schedule_relationship",
        "scrape_id",
        "current_date",
        "stop_headsign",
        "pickup_type",
        "drop_off_type",
        "shape_dist_traveled",
        "timepoint",
        "stop_lat",
        "stop_code",
        "stop_lon",
        "parent_station",
        "stop_desc",
        "stop_name",
        "location_type",
        "zone_id",
        "block_id",
        "trip_headsign",
        "shape_id",
        "service_id",
        "trip_short_name",
        "end_date",
        "monday",
        "tuesday",
        "wednesday",
        "thursday",
        "friday",
        "saturday",
        "sunday",
        "route_long_name",
        "route_type",
        "agency_id",
        "route_desc",
        "route_short_name",
    ]

    selected_realtime_df = realtime_df_enriched.select(
        realtime_selected_columns
    ).with_columns(
        current_datetime_three_hour_bucket=pl.col("current_datetime").dt.truncate("3h")
    )

    merged_df = selected_realtime_df.join(
        weather_df,
        left_on="current_datetime_three_hour_bucket",
        right_on="nowobsTime_three_hour_bucket",
        how="left",
        coalesce=True,
    ).join_asof(
        position_df,
        by="trip_id",  # Use 'by' for the non-time-based matching column
        left_on="current_datetime",  # Time column to match on
        right_on="current_datetime",  # Time column to match on
        strategy="nearest",
        tolerance="3m",
        coalesce=False,
    )

    return merged_df

In [17]:
merged_translink_df = merge_dataframes_etl()

In [18]:
merged_translink_df_full = merged_translink_df.collect()

In [19]:
merged_translink_df_full

is_deleted,trip_id,start_date,schedule_relationship,route_id,direction_id,vehicle_id,vehicle_label,current_datetime,hour,minute,stop_sequence,stop_id,arrival_delay,last_arrival_delay,last_arrival_delay_time,last_record_rank,arrival_time,scheduled_arrival_time,departure_delay,departure_time,scheduled_departure_time,stop_schedule_relationship,scrape_id,current_date,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_lat,stop_code,stop_lon,parent_station,stop_desc,stop_name,location_type,zone_id,block_id,trip_headsign,shape_id,service_id,trip_short_name,end_date,monday,tuesday,wednesday,thursday,friday,saturday,sunday,route_long_name,route_type,agency_id,route_desc,route_short_name,current_datetime_three_hour_bucket,nowtemp,nowtext,nowprecip,current_stop_sequence,vehicle_latitude,vehicle_longitude,current_datetime_right
bool,str,str,i64,str,i64,str,str,"datetime[ns, America/Vancouver]",i8,i8,i64,str,i64,i64,"datetime[ns, America/Vancouver]",i64,"datetime[μs, America/Vancouver]",str,f64,"datetime[μs, America/Vancouver]",str,i64,str,date,str,i64,str,f64,i64,f64,i64,f64,i64,str,str,i64,str,str,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,"datetime[ns, America/Vancouver]",i64,str,f64,i64,f64,f64,"datetime[ns, America/Vancouver]"
false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,19,34,1,"""11251""",0,0,2025-01-09 19:49:49.052991 PST,4,2025-01-09 19:52:00 PST,"""19:52:00""",0.0,2025-01-09 19:52:00 PST,"""19:52:00""",0,"""9ef7096c1db33bd1""",2025-01-09,null,null,null,null,1,49.209087,61337,-123.116823,null,null,"""Marine Drive Station @ Bay 1""",0,"""BUS ZN""","""b_2117689""","""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,"""TL""",null,"""003""",2025-01-09 18:00:00 PST,7,"""Light Rain""",3.0,null,null,null,null
false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,19,34,2,"""2151""",-15,26,2025-01-09 19:54:48.676472 PST,5,2025-01-09 19:54:50 PST,"""19:55:05""",-15.0,2025-01-09 19:54:50 PST,"""19:55:05""",0,"""9ef7096c1db33bd1""",2025-01-09,null,null,null,0.7837,0,49.212198,52131,-123.109277,null,null,"""Eastbound SW Marine Dr @ Manit…",0,"""BUS ZN""","""b_2117689""","""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,"""TL""",null,"""003""",2025-01-09 18:00:00 PST,7,"""Light Rain""",3.0,null,null,null,null
false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,19,34,3,"""2152""",-34,7,2025-01-09 19:54:48.676472 PST,5,2025-01-09 19:55:29 PST,"""19:56:03""",-34.0,2025-01-09 19:55:29 PST,"""19:56:03""",0,"""9ef7096c1db33bd1""",2025-01-09,null,null,null,1.0315,0,49.21214,52132,-123.106006,null,null,"""Eastbound SE Marine Dr @ Ontar…",0,"""BUS ZN""","""b_2117689""","""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,"""TL""",null,"""003""",2025-01-09 18:00:00 PST,7,"""Light Rain""",3.0,null,null,null,null
false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,19,34,4,"""142""",-10,31,2025-01-09 19:54:48.676472 PST,5,2025-01-09 19:57:18 PST,"""19:57:28""",-10.0,2025-01-09 19:57:18 PST,"""19:57:28""",0,"""9ef7096c1db33bd1""",2025-01-09,null,null,null,1.3937,0,49.211565,50142,-123.102035,null,null,"""Northbound Main St @ SE Marine…",0,"""BUS ZN""","""b_2117689""","""3 Main/To Waterfront Station""",296959,"""1_merged_14516836""",null,20250420,1,1,1,1,1,0,0,"""Main/Waterfront Station""",3,"""TL""",null,"""003""",2025-01-09 18:00:00 PST,7,"""Light Rain""",3.0,null,null,null,null
false,"""14252592""","""20250109""",0,"""6613""",0,"""2537""","""2537""",2025-01-09 19:34:48.829195 PST,19,34,5,"""144""",-28,13,2025-01-09 19:54:48.676472 PST,5,2025-01-09 19:58:19 PST,"""19:58:47""",-28.0

In [20]:
merged_translink_df_full.write_parquet(
    "../data/processed/merged_translink_df_full.parquet",
)

### Small EDA

In [21]:
# realtime_df_enriched.filter(
#     # pl.col("stop_name") == "Northbound Dunbar St @ W King Edward Ave",
#     # pl.col("route_id") == "6627",
#     # pl.col('arrival_time').dt.hour() == 10,
#     pl.col("trip_id") == "14007354",
#     pl.col("current_date") == datetime(2024, 12, 31),
#     pl.col("stop_sequence").is_between(40, 50),
# ).sort("stop_sequence").filter(pl.col("last_record_rank") == 1).collect()

In [22]:
# example_25_all = (
#     realtime_df_enriched.filter(
#         pl.col("route_id") == "6627",
#         pl.col("direction_id") == 1,
#         pl.col("current_date") == datetime(2024, 12, 28),
#     )
#     .sort("stop_sequence")
#     .filter(pl.col("last_record_rank") == 1)
#     .collect()
#     .to_pandas()
# )

In [23]:
# example_25_all.head()

In [24]:
# import matplotlib.pyplot as plt
# import pandas as pd
# import seaborn as sns

# def plot_delays(df):
#     # Create figure and axis
#     plt.figure(figsize=(12, 6))

#     # Get unique dates for different colors
#     unique_dates = df['trip_id'].unique()

#     # Create color palette
#     colors = sns.color_palette("husl", n_colors=len(unique_dates))

#     # Plot each date's data as a separate line
#     for date, color in zip(unique_dates, colors):
#         mask = df['trip_id'] == date
#         data = df[mask]
#         plt.plot(data['stop_sequence'],
#                 data['last_arrival_delay'],
#                 color=color,
#                 label=str(date))

#     # Customize the plot
#     plt.xlabel('Stop Sequence')
#     plt.ylabel('Last Arrival Delay')
#     plt.title('Arrival Delays by Stop Sequence')
#     plt.legend(title='Date', bbox_to_anchor=(1.05, 1), loc='upper left')
#     plt.grid(True, linestyle='--', alpha=0.7)

#     # Adjust layout to prevent legend cutoff
#     plt.tight_layout()

#     return plt

# plot_delays(example_25_all)

In [25]:
# selection = alt.selection_point(fields=['trip_id'], bind='legend')
# alt.Chart(example_25_all).mark_line().encode(
#     x='stop_sequence',
#     y='last_arrival_delay',
#     color='trip_id:N',
#     tooltip=['trip_id', 'stop_sequence', 'last_arrival_delay','hour'],
#     opacity=alt.when(selection).then(alt.value(1)).otherwise(alt.value(0.2))
# ).add_params(
#     selection
# )

In [26]:
# realtime_df_enriched.filter(
#     # pl.col("stop_name") == "Northbound Dunbar St @ W King Edward Ave",
#     pl.col("route_id") == "6627",
#     # pl.col('arrival_time').dt.hour() == 10,
#     pl.col("hex_id") == "e9e1cca1cb5db0e8",
#     pl.col('direction_id') == 0
# ).sort('stop_sequence').limit(10).collect()

In [27]:
# realtime_df_enriched.filter(pl.col("last_record_rank") == 1).sort('current_datetime',descending = True).limit(10).collect()